# B8: our full TinyLlama model

**Goal:** trained weights -> our own complete forward -> HF-matching logits and greedy tokens in FP32.

```
IDs (N,) -> embedding (N,2048) -> 22 decoder layers
         -> final RMSNorm -> lm_head -> logits (N,32000)
```

HF supplies the tokenizer, weights and independent reference. Our forward uses only the modules we built; it does not call HF model code.

**Spec:** `model(input_ids, positions, AttentionMetadata)` returns logits `(N, vocab)` in FP32. Input IDs and positions are flattened `(N,)`; metadata positions must agree. Strict checkpoint loading, no KV cache.

Run top to bottom in **Colab on a T4**. Two FP32 model copies need about **8.8 GB for weights** plus temporary memory. Close previous GPU-heavy notebook kernels first.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)


In [ ]:
import importlib
import torch
import tinyserve.models.llama, tinyserve.probe
for module in (tinyserve.models.llama, tinyserve.probe):
    importlib.reload(module)
from tinyserve.models.llama import LlamaForCausalLM
from tinyserve.models.attention import AttentionMetadata
from tinyserve.basics import load_model, encode
from tinyserve.envinfo import collect_env, save_result
from tinyserve.probe import compare_full_model, compare_greedy_steps

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
print(collect_env())


## 1. Test first

Run the model-component tests B2-B8. B8 adds complete-stack HF parity, strict loading, tied/untied weights, final norm and LM head wiring, packed isolation, causality and exact greedy agreement with HF `generate(use_cache=False)` on a small model.

A failure stops the notebook. Read skips: HF, SDPA, plotting and CUDA tests should be available here.

Real full-stack FP32 logits use `rtol=1e-5, atol=1e-4` for accumulated rounding. Token equality is a separate required assertion. A synthetic FP16 test compares against the same rounded weights in FP32, not bit equality.

In [ ]:
test_files = ["tests/test_b2_rmsnorm.py", "tests/test_b3_rope.py", "tests/test_b4_attention.py", "tests/test_b5_gqa.py",
              "tests/test_b6_mlp.py", "tests/test_b7_decoder.py", "tests/test_b8_model.py"]
test_run = subprocess.run([sys.executable, "-m", "pytest", *test_files, "-v", "-ra"], capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()


## 2. Strictly load all trained weights

The names match HF: `model.embed_tokens`, `model.layers.0...21`, `model.norm`, `lm_head`. `strict=True` rejects missing, extra or incorrectly shaped weights. This copies weights into **our architecture**, not an HF wrapper.

TinyLlama does not tie its input/output embeddings: there are two separate 32000-by-2048 matrices. Their jobs differ: lookup a token's initial vector, versus score every possible next token.

In [ ]:
tokenizer, hf_model = load_model(device=DEVICE, dtype=torch.float32)
hf_model.set_attn_implementation("eager")
our_model = LlamaForCausalLM.from_huggingface(hf_model)
our_params = sum(parameter.numel() for parameter in our_model.parameters())
hf_params = sum(parameter.numel() for parameter in hf_model.parameters())
assert our_params == hf_params
assert len(our_model.model.layers) == 22
print(f"parameters: ours {our_params:,}, HF {hf_params:,}")
print("weight tensors:", len(our_model.state_dict()))
print("tied embeddings:", our_model.lm_head.weight is our_model.model.embed_tokens.weight)


## 3. Full logits on multiple prompts

B7 used HF's input for each isolated layer. Here our layer 0 feeds our layer 1, and so on through all 22 layers, the final norm and LM head.

Compare **every logits row** and its argmax ID. A numerical tolerance pass alone is insufficient: two nearly tied candidates can switch places. Record the top-two margin on the last row to see how close the candidates are.

In [ ]:
prompts = ["The capital of France is", "The capital of Germany is", "A tiny language model can"]
prompt_rows = []
for prompt_text in prompts:
    prompt_ids = encode(tokenizer, prompt_text, DEVICE)
    result = compare_full_model(our_model, hf_model, prompt_ids)
    result["prompt"] = prompt_text
    prompt_rows.append(result)
    print(f"{prompt_text!r}: max |diff| {result['max_abs_diff']:.2e}, "
          f"all argmax equal {result['argmax_equal']}, next {tokenizer.decode([result['next_token_id']])!r}, "
          f"HF top-two margin {result['hf_top_two_margin']:.4f}")


## 4. Flattened packed requests

Concatenate two prompts, restart positions per request and assign distinct sequence IDs. Run one flattened forward, then compare with independent forwards. The model must not let one request influence another.

In [ ]:
first_ids = encode(tokenizer, prompts[0], DEVICE)
second_ids = encode(tokenizer, prompts[1], DEVICE)
packed_ids = torch.cat([first_ids, second_ids])
first_positions = torch.arange(first_ids.numel(), device=DEVICE)
second_positions = torch.arange(second_ids.numel(), device=DEVICE)
packed_positions = torch.cat([first_positions, second_positions])
packed_sequence_ids = torch.cat([torch.zeros_like(first_ids), torch.ones_like(second_ids)])
with torch.no_grad():
    packed_logits = our_model(packed_ids, packed_positions, AttentionMetadata(packed_positions, packed_sequence_ids))
    separate_logits = torch.cat([
        our_model(first_ids, first_positions, AttentionMetadata(first_positions)),
        our_model(second_ids, second_positions, AttentionMetadata(second_positions)),
    ])
packed_max_diff = (packed_logits - separate_logits).abs().max().item()
torch.testing.assert_close(packed_logits, separate_logits, rtol=1e-5, atol=1e-4)
assert torch.equal(packed_logits.argmax(-1), separate_logits.argmax(-1))
print("packed vs separate max difference:", packed_max_diff)


## 5. Token-exact greedy parity, step by step

For each prompt, run both models on the same growing prefix. Check logits and argmax at every step, append the agreed token, and stop on EOS or 12 new tokens. A mismatch raises immediately.

This is a **verification loop**, not the public generation API (C2). No KV cache, no sampling, no speed claims. Raw text prompts can continue into lists instead of giving concise chat answers; we care about matching HF, not the prose style.

In [ ]:
greedy_rows = []
for prompt_text in prompts:
    prompt_ids = encode(tokenizer, prompt_text, DEVICE)
    generated_ids, step_rows = compare_greedy_steps(our_model, hf_model, prompt_ids, max_new_tokens=12)
    generated_text = tokenizer.decode(generated_ids.tolist(), skip_special_tokens=True)
    greedy_rows.append({"prompt": prompt_text, "token_ids": generated_ids.tolist(), "text": generated_text,
                        "new_tokens": len(step_rows), "steps": step_rows})
    print(f"{len(step_rows)} matching steps: {generated_text!r}")


## 6. Save results and three interview questions

1. **Why full-stack parity?** It checks our own layer outputs feeding later layers, not isolated blocks on HF inputs.
2. **Why logits tolerance plus exact tokens?** Small score errors can change argmax near a tie.
3. **What does HF still do?** Supply trained weights/tokenizer and the test oracle. Our forward computes the entire model.

Error ranges below are across prompts and greedy steps, not performance trials. Save the notebook with outputs and retain this JSON.

**Next: C1 sampling** (greedy, temperature, top-k, top-p), then C2 generation and C3 KV cache.

In [ ]:
all_differences = [row["max_abs_diff"] for row in prompt_rows]
all_differences += [step["max_abs_diff"] for row in greedy_rows for step in row["steps"]]
saved_path = save_result("B8_full_model", {
    "dtype": "float32", "backend": "reference", "hf_backend": "eager", "use_cache": False,
    "num_params": our_params,
    "prompt_comparison": prompt_rows,
    "packed_max_abs_diff": packed_max_diff,
    "greedy_comparison": greedy_rows,
    "diff_range_across_prompts_and_steps": {"min": min(all_differences), "max": max(all_differences)},
    "pytest": {"returncode": test_run.returncode, "stdout": test_run.stdout, "stderr": test_run.stderr},
})
print(saved_path)
print(saved_path.read_text())


In [ ]:
import gc
del our_model, hf_model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still held by this kernel: {torch.cuda.memory_allocated() / 2**20:.0f} MiB")
